In [1]:
import uuid
from typing import Any, Dict, Tuple

from langgraph.store.memory import InMemoryStore


def create_store() -> InMemoryStore:
    return InMemoryStore()


def store_memory(store: InMemoryStore, namespace: Tuple[str, str], memory_id: str, memory_data: Dict[str, Any]) -> None:
    store.put(namespace, memory_id, memory_data)


def retrieve_memories(store: InMemoryStore, namespace: Tuple[str, str]) -> list:
    return store.search(namespace)


in_memory_store = create_store()
user_id = "1"
namespace = (user_id, "memories")
memory_id = str(uuid.uuid4())
memory_data = {"food_preference": "我喜欢苹果"}

store_memory(in_memory_store, namespace, memory_id, memory_data)
print(f"已存储记忆 ID: {memory_id}")

memories = retrieve_memories(in_memory_store, namespace)
print(f"\n命名空间 '{namespace}' 中的记忆数量: {len(memories)}")

if memories:
    print("\n最新记忆内容")
    print("-" * 50)
    latest_memory = memories[-1]
    if hasattr(latest_memory, "dict"):
        print(latest_memory.dict())
    else:
        print(latest_memory)

已存储记忆 ID: 9fee2df2-bdae-4fd0-9457-e73ad202f31f

命名空间 '('1', 'memories')' 中的记忆数量: 1

最新记忆内容
--------------------------------------------------
{'namespace': ['1', 'memories'], 'key': '9fee2df2-bdae-4fd0-9457-e73ad202f31f', 'value': {'food_preference': '我喜欢苹果'}, 'created_at': '2026-07-05T15:24:12.255949+00:00', 'updated_at': '2026-07-05T15:24:12.255953+00:00', 'score': None}


In [7]:
import os
import uuid
from typing import Any, Dict, Tuple

from langchain_community.embeddings import DashScopeEmbeddings
from langgraph.store.memory import InMemoryStore

embeddings = DashScopeEmbeddings(model="text-embedding-v4", dashscope_api_key=os.getenv("QWEN_API"))


def create_store() -> InMemoryStore:
    return InMemoryStore(index={"embed": embeddings, "dims": 1536})


def store_memory(store: InMemoryStore, namespace: Tuple[str, str], memory_id: str, memory_data: Dict[str, Any]) -> None:
    store.put(namespace, memory_id, memory_data)


def retrieve_memories(store: InMemoryStore, namespace: Tuple[str, str]) -> list:
    return store.search(namespace)


in_memory_store = create_store()
user_id = "1"
namespace = (user_id, "memories")
memory_id = str(uuid.uuid4())

store_memory(in_memory_store, namespace, str(uuid.uuid4()), {"food_preference": "苹果是我喜欢吃的水果"})
store_memory(in_memory_store, namespace, str(uuid.uuid4()), {"food_preference": "橘子是我喜欢吃的水果"})
store_memory(in_memory_store, namespace, str(uuid.uuid4()), {"pet_preference": "猫是我喜欢的动物"})


memories = retrieve_memories(in_memory_store, namespace)
print(f"\n命名空间 '{namespace}' 中的记忆数量: {len(memories)}")

print(in_memory_store.search(namespace, query="我喜欢吃什么", limit=2))
print(in_memory_store.search(namespace, query="我喜欢的宠物", limit=2))


命名空间 '('1', 'memories')' 中的记忆数量: 3
[Item(namespace=['1', 'memories'], key='f94b73df-c311-4db1-b514-315d6806dda9', value={'food_preference': '苹果是我喜欢吃的水果'}, created_at='2026-07-05T15:34:09.484665+00:00', updated_at='2026-07-05T15:34:09.484680+00:00', score=0.6281118593807559), Item(namespace=['1', 'memories'], key='b1b535f9-e393-45d4-96ea-24a4708ddd6c', value={'food_preference': '橘子是我喜欢吃的水果'}, created_at='2026-07-05T15:34:09.787865+00:00', updated_at='2026-07-05T15:34:09.787869+00:00', score=0.5689497776862256)]
[Item(namespace=['1', 'memories'], key='94ca348e-93e1-4d0a-ae45-3e6baa5c58e8', value={'pet_preference': '猫是我喜欢的动物'}, created_at='2026-07-05T15:34:10.088946+00:00', updated_at='2026-07-05T15:34:10.088950+00:00', score=0.6846358865356471), Item(namespace=['1', 'memories'], key='f94b73df-c311-4db1-b514-315d6806dda9', value={'food_preference': '苹果是我喜欢吃的水果'}, created_at='2026-07-05T15:34:09.484665+00:00', updated_at='2026-07-05T15:34:09.484680+00:00', score=0.5261216528423948)]
